In [1]:
!pip install GEOparse pandas scikit-learn gseapy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.0/693.0 kB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [GEOparse]


In [2]:
import GEOparse, pandas as pd

gse = GEOparse.get_GEO("GSE76275", destdir="./")
expr = gse.pivot_samples("VALUE")      # probes x samples
print(expr.shape)                      # expect about (54675, 265)

21-Nov-2026 14:10:15 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
21-Nov-2026 14:10:15 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE76nnn/GSE76275/soft/GSE76275_family.soft.gz to ./GSE76275_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE76nnn/GSE76275/soft/GSE76275_family.soft.gz to ./GSE76275_family.soft.gz
100%|██████████| 129M/129M [00:01<00:00, 118MB/s]
21-Nov-2026 14:10:16 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
21-Nov-2026 14:10:16 DEBUG downloader - Moving /tmp/tmpcrezjlle to /content/GSE76275_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpcrezjlle to /content/GSE76275_family.soft.gz
21-Nov-2026 14:10:17 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE76nnn/GSE76275/soft/GSE76275_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE76nnn/GSE76275

(54675, 265)


In [3]:
ids = [int(s.replace("GSM", "")) for s in expr.columns]
labels = []
for i in ids:
    if 1974566 <= i <= 1974763:
        labels.append(1)    # TNBC
    elif 1978883 <= i <= 1978949:
        labels.append(0)    # non-TNBC
    else:
        labels.append(None)

y = pd.Series(labels, index=expr.columns, name="label")
print(y.value_counts(dropna=False))    # expect 1: 198, 0: 67, no None


label
1    198
0     67
Name: count, dtype: int64


In [4]:
g = gse.gsms[expr.columns[0]]
print(g.metadata["title"], g.metadata["characteristics_ch1"])
g = gse.gsms[expr.columns[-1]]
print(g.metadata["title"], g.metadata["characteristics_ch1"])

['S1-H10'] ['tissue: Breast cancer', 'gender: Female', 'race: Caucasian', 'body mass index: 32', 'menopausal status: Post-Menopausal', 'histology group: Infiltrating Ductal Carcinoma', 'histology: Infiltrating Ductal Carcinoma', 'ajcc stage (7th edition, 2010): T2N1M0', 'set: Validation TN', 'er: Negative', 'pr: Negative', 'her2: Negative', 'triple-negative status: TN', 'tumor size: 2 - 5 cm', 'positive nodes: 1 - 3', 'metastases: No mets', 'tnbc subtype: Mesenchymal (MES)']
['S4-H280'] ['tissue: Breast cancer', 'triple-negative status: not TN', 'age (years): 71', 'gender: Female', 'race: Caucasian', 'body mass index: 27', 'menopausal status: Post-menopausal', 'histology: Infiltrating ductal carcinoma of the breast', 'ajcc stage (7th edition, 2010): T1NXM0', 'set: Validation not TN', 'er: Positive', 'pr: Positive', 'her2: Negative', 'tumor grade: Moderately Differentiated']


In [5]:
data = expr.T
data["label"] = y
data = data.dropna(subset=["label"])
data.to_csv("tnbc_data.csv")
print(data.shape)

(265, 54676)
